# Section 1 - Loading Data, Models and Labels

## 1.1 Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import random
import torch
import joblib
import json
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, average_precision_score


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:
from utils.architect_classes import LSTMModel, TemporalBlock, TCNModel, TapNetEncoder

if "LSTMModel" in dir() and "TCNModel" in dir():
    print("Classes imported successfully")
else:
    print("Import failed — check architect_classes.py")

Classes imported successfully


## 1.2 Mount and Setup

In [ ]:
from google.colab import drive
import sys
import os

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MOUNT_DIR = '/content/drive/MyDrive/SCANIA Component X/Results/'
MODEL_DIR = MOUNT_DIR + 'Models/'
PRED_DIR = MOUNT_DIR + 'Predictions/'
DATA_DIR = '/content/drive/MyDrive/SCANIA Component X/Data/Processed Data/'



print(f"Data found at: {MOUNT_DIR}")
print(f"Files: {os.listdir(MOUNT_DIR)}")

print("__________________________________")
print(f"Models found at: {MODEL_DIR}")
print(f"Files: {os.listdir(MODEL_DIR)}")

print("__________________________________")
print(f"Models found at: {PRED_DIR}")
print(f"Files: {os.listdir(PRED_DIR)}")

print("__________________________________")
print(f"Models found at: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")

Data found at: /content/drive/MyDrive/SCANIA Component X/Results/
Files: ['Predictions', 'Models']
__________________________________
Models found at: /content/drive/MyDrive/SCANIA Component X/Results/Models/
Files: ['lstm_model_best.pt', 'tapnet_embed.pt', 'lstm_best.pt', 'tcn_best.pt', 'smote_lr_model.joblib', 'smote_xgb_model.joblib', 'tapnet_encoder.pt', 'lr_model.joblib', 'hyperparams.json']
__________________________________
Models found at: /content/drive/MyDrive/SCANIA Component X/Results/Predictions/
Files: ['X_train_embedded.npy', 'X_val_embedded.npy', 'lstm_val_predictions.csv', 'lr_val_predictions.csv', 'tcn_val_predictions.csv', 'X_test_embedded.npy', 'X_resampled.npy', 'y_resampled.npy']
__________________________________
Models found at: /content/drive/MyDrive/SCANIA Component X/Data/Processed Data/
Files: ['train_processed.csv', 'val_processed.csv', 'scaler.joblib', 'test_processed.csv', 'val_labels_original.csv', 'test_labels_original.csv']


## 1.4 Loading the Models




In [ ]:
# Load hyperparameters
with open(MODEL_DIR + 'hyperparams.json', 'r') as f:
    hp = json.load(f)

# LR Model (sklearn — uses joblib)
lr_model = joblib.load(MODEL_DIR + 'lr_model.joblib')

# LSTM Model (PyTorch — recreate architecture, load weights)
lstm_model = LSTMModel(
    input_size=hp['INPUT_SIZE'],
    hidden_size=hp['LSTM']['hidden_size'],
    num_layers=hp['LSTM']['num_layers'],
    dropout=hp['LSTM']['dropout']
)
lstm_model.load_state_dict(torch.load(MODEL_DIR + 'lstm_best.pt', map_location=DEVICE))
lstm_model.to(DEVICE)
lstm_model.eval()

# TCN Model (PyTorch — same pattern)
tcn_model = TCNModel(
    input_size=hp['INPUT_SIZE'],
    num_channels=hp['TCN']['num_channels'],
    num_layers=hp['TCN']['num_layers'],
    kernel_size=hp['TCN']['kernel_size'],
    dropout=hp['TCN']['dropout']
)
tcn_model.load_state_dict(torch.load(MODEL_DIR + 'tcn_best.pt', map_location=DEVICE))
tcn_model.to(DEVICE)
tcn_model.eval()

# SMOTE models (sklearn — uses joblib)
smote_lr = joblib.load(MODEL_DIR + 'smote_lr_model.joblib')
smote_xgb = joblib.load(MODEL_DIR + 'smote_xgb_model.joblib')
X_test_embedded = np.load(PRED_DIR + 'X_test_embedded.npy')


print("All models loaded.")

All models loaded.


### 1.5 Loading the Labels

In [ ]:
test_labels_original = pd.read_csv(DATA_DIR + 'test_labels_original.csv')
print(test_labels_original['class_label'].value_counts().sort_index())


class_label
0    4903
1      26
2      15
3      41
4      60
Name: count, dtype: int64


## 1.6 Loading the Data

In [ ]:
# Loading the Test data
test_data = pd.read_csv(DATA_DIR + 'test_processed.csv')

test_data.head()

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,397_27,397_28,397_29,397_30,397_31,397_32,397_33,397_34,397_35,binary_label
0,1,4.4,0.000000,0.000000,0.000000,0.000000,0.000000,0.003435,0.000038,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,1,5.0,0.030175,0.080517,0.025878,0.094354,0.000439,0.033976,0.010404,0.006605,...,0.128555,0.118170,0.000000,0.028971,0.050642,0.093097,0.066555,0.015008,0.000000,0
2,1,8.2,0.269060,0.873435,0.239978,0.658122,1.575000,0.061406,0.053288,0.089010,...,0.641734,0.476011,0.160920,0.315054,0.704811,0.917932,0.873239,0.532427,0.024221,0
3,1,9.8,-0.173679,-0.565370,-0.147491,-0.042460,-0.866667,-0.044198,-0.024575,-0.064213,...,-0.408132,-0.328076,0.160920,-0.148474,-0.439985,-0.553251,-0.456478,-0.186912,0.048443,0
4,1,20.6,0.586860,1.553454,0.519557,2.224209,-0.224561,0.178215,0.144977,0.153923,...,1.194860,1.629276,1.057471,0.989136,1.037613,1.505081,1.515538,1.373240,2.017301,0


## 1.7 Making the Cost Matrix
The cost Matrix is described [here](https://www.nature.com/articles/s41597-025-04802-6), we are simply making the matrix based on the information in the article



In [ ]:
cost_matrix = pd.DataFrame(
    [[0, 200, 300, 400, 500],     # First Column,   predicted = 0
    [7, 0, 200, 300, 400],        # Second Column,  predicted = 1
    [8, 7, 0, 200, 300],          # Third Column,   predicted = 2
    [9, 8, 7, 0, 200],            # Fourth Column,  predicted = 3
    [10, 9, 8, 7, 0]],            # Fifth Column,   predicted = 4
    index =['Pred: 0', 'Pred: 1', 'Pred: 2', 'Pred: 3', 'Pred: 4'],
    columns=['Actual: 0', 'Actual: 1', 'Actual: 2', 'Actual: 3', 'Actual: 4']
).T


print(cost_matrix)

           Pred: 0  Pred: 1  Pred: 2  Pred: 3  Pred: 4
Actual: 0        0        7        8        9       10
Actual: 1      200        0        7        8        9
Actual: 2      300      200        0        7        8
Actual: 3      400      300      200        0        7
Actual: 4      500      400      300      200        0
